In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/O.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/S.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed42_c.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed2026_a_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-1.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/I.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_Best_E_fallback.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-

## Setup & Data Loading

In [2]:
import pandas as pd
 
COMP = '/kaggle/input/competitions/playground-series-s6e4/'
DS   = '/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/'
 
sub = pd.read_csv(COMP + 'sample_submission.csv')
 
# Load files 
df8 = pd.read_csv(DS + 'Q.csv').rename(columns={'Irrigation_Need':'df8'})  
df5 = pd.read_csv(DS + 'U.csv').rename(columns={'Irrigation_Need':'df5'}) 
df3 = pd.read_csv(DS + 'S.csv').rename(columns={'Irrigation_Need':'df3'})  
df4 = pd.read_csv(DS + 'T.csv').rename(columns={'Irrigation_Need':'df4'})  
df7 = pd.read_csv(DS + 'W.csv').rename(columns={'Irrigation_Need':'df7'})  
df9 = pd.read_csv(DS + 'Z.csv').rename(columns={'Irrigation_Need':'df9'}) 
 
# Merge
dfs = df8.copy()
for d in [df5, df3, df4, df7, df9]:
    dfs = dfs.merge(d, on='id')
 
N = ['df8','df5','df3','df4','df7','df9']
 
def make_pattern(row, cols):
    parts = []
    for c in cols:
        v = row[c][0]
        if v == 'M': v = '_'
        parts.append(v + ' ')
    return ''.join(parts)
 
dfs['wh'] = dfs.apply(lambda x: make_pattern(x, N), axis=1)
print("Pattern distribution (top 10):")
print(dfs['wh'].value_counts().head(10))
print(f"\nTotal rows: {len(dfs):,}")
agree = (dfs[N].nunique(axis=1) == 1)
print(f"All 6 agree: {agree.sum():,}  Disagree: {(~agree).sum():,}")

Pattern distribution (top 10):
wh
L L L L L L     159251
_ _ _ _ _ _      99556
H H H H H H       9843
H _ H _ _ _        226
_ _ H _ _ _        182
_ H _ H H H        166
_ L L L L L        138
H _ _ _ _ _        132
_ _ L _ _ _        114
L _ L _ _ _         85
Name: count, dtype: int64

Total rows: 270,000
All 6 agree: 268,650  Disagree: 1,350


In [3]:
def schema10(x, lns):
    if x['wh'] == 'L _ L _ _ _ ': return 'Low'     
    if x['wh'] == '_ L _ L L L ': return 'Medium'  
    first5 = lns[:-1]  
    last   = lns[-1]  
    if x[first5[0]]==x[first5[1]]==x[first5[2]]==x[first5[3]]==x[first5[4]]:
        return x[first5[0]]
    return x[last] 
 
dfv10 = sub.copy()
dfv10['Irrigation_Need'] = dfs.apply(lambda x: schema10(x, N), axis=1)
print(f"\nStep1 (schema10) dist: {dfv10['Irrigation_Need'].value_counts().to_dict()}")

def return_back(x):
    if x['wh'] == 'H _ H _ _ _ ': return 'Medium'  
    if x['wh'] == '_ H _ H H H ': return 'High'    
    if x['wh'] == 'H _ _ _ _ _ ': return 'Medium'  
    return x['df8']  
 
dfs['Irrigation_Need'] = dfs.apply(lambda x: return_back(x), axis=1)
sub134 = sub.copy()
sub134['Irrigation_Need'] = dfs['Irrigation_Need']
print(f"Step2 (return_back) dist: {sub134['Irrigation_Need'].value_counts().to_dict()}")
sub134.to_csv('submission_blendmix.csv', index=False)
print("Saved: submission_blendmix.csv")
 
dfv10_col = dfv10.rename(columns={'Irrigation_Need':'dfv10'})
dfs2 = dfs.merge(dfv10_col, on='id')
 
def correct(x):
    if x['wh'] == '_ H H H H H ': return 'Medium'  
    return x['dfv10'] 
 
dfs2['Irrigation_Need'] = dfs2.apply(lambda x: correct(x), axis=1)
sub145 = sub.copy()
sub145['Irrigation_Need'] = dfs2['Irrigation_Need']
print(f"Step3 (correct) dist: {sub145['Irrigation_Need'].value_counts().to_dict()}")
sub145.to_csv('submission_topblend', index=False)
print("Saved: submission_topblend.csv")
 
print("\n" + "="*55)


Step1 (schema10) dist: {'Low': 159516, 'Medium': 100332, 'High': 10152}
Step2 (return_back) dist: {'Low': 159408, 'Medium': 100502, 'High': 10090}
Saved: submission_blendmix.csv
Step3 (correct) dist: {'Low': 159516, 'Medium': 100394, 'High': 10090}
Saved: submission_topblend.csv

